In [3]:
import base64
import requests
import os
import json
from PIL import Image


api_base = "https://neudm.zeabur.app/v1"
api_key = "sk-T05m0OqxOgKUjErs8c231e1c02E24573A17977F5E839E91c"
image_path = "data/images/normal"
json_path = "normal"
output_path = "data/res_summary_len8"
# with open('data/result/normal/n1058.json', 'r') as file:
#     QA_pairs = json.load(file)
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def get_image_summary(base64_image, json_data=None):

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }

    # content_system = (
    #     "You are a professional network topology analysis expert."
    #     "Your goal is to accurately and clearly analyze and summarize a topology image\n"
    #     "You will be provided two information:"
    #     " 1.A topology image "
    #     " 2. (Opitional) QA pairs about the main elements of the topology\n"
    #     "Please follow these steps:"
    #     "1. Image Analysis: Carefully examine the topology image, identifying key nodes and their connections."
    #     "2. (Opitional) QA- pairs analysis: Try to understand the meaning of the provided QA pairs and their relevance to the topology."
    #     "3. Description Method Selection:** Choose a **layered description** (suitable for topologies with clear hierarchies, such as core and access layers) or a **distributed description** (suitable for topologies with scattered nodes) based on the observed structure."
    #     "4. Summary following the description method: follow the description method to generate a summary of the topology."
    #     "Output only a English summary of the topology image without any analysis. "
    #     "Output format: This is a ... topo image. The main elements are ... . <Describe the whole structure and function>, <Describe the details>"
    # )
    content_system = (
        "You are a professional network topology analysis expert."
        "Your goal is to accurately and clearly analyze and summarize a topology image\n"
        "You will be provided two information:"
        "1. A topology image "
        "2. (Optional) QA pairs related to the main elements of the topology. \n"
        "Please follow these steps: "
        "1. Image Analysis: Examine the topology image closely, identifying key nodes and their connections. "
        "2. QA Pairs Analysis: If provided, interpret the QA pairs to understand their relevance to the topology. "
        "3. Description Method Selection:** Choose a **layered description** (suitable for topologies with clear hierarchies, such as core and access layers) or a **distributed description** (suitable for topologies with scattered nodes) based on the observed structure."
        "4. Generate Summary: Using the chosen description method, create a summary of the topology. "
        "Output only an English summary of the topology image without any further analysis. The format should be: "
        "Output format: This network topology represents a [type] architecture. Key components include [list key nodes or devices], connected via [describe primary connection types, e.g., switches, routers]. The overall structure supports [describe the purpose, e.g., data flow, redundancy]. Notable features include [mention any special attributes, e.g., high availability, scalability], ensuring efficient communication between [describe user groups or services]......"
    )

    messages = [
        {"role": "system", "content": content_system},
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"<Topo_image>, QA pairs: {json_data}.",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                },
            ],
        },
    ]

    payload = {
        "model": "gpt-4o-mini",
        "messages": messages,
        "temperature": 0.2,
        # "max_tokens": 300,
    }
    try:
        response = requests.post(
            f"{api_base}/chat/completions", headers=headers, json=payload
        )
        response.raise_for_status()  
        result = response.json()
        summary = result["choices"][0]["message"]["content"]
        return summary
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        print(f"Response content: {response.text}")
        return None
    except (KeyError, IndexError, json.JSONDecodeError) as e:
        print(f"Error processing response: {e}")
        return None


# 遍历图片目录并处理每个图片
for filename in os.listdir(image_path):
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp")):
        image_filepath = os.path.join(image_path, filename)
        json_filename = os.path.splitext(filename)[0] + ".json"
        json_filepath = os.path.join(json_path, json_filename)

        # 读取JSON文件内容（如果存在）
        json_data = None
        if os.path.exists(json_filepath):
            try:
                with open(json_filepath, "r", encoding="utf-8") as json_file:
                    json_data = json.load(json_file)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON from {json_filepath}: {e}")
                continue

        # 编码图片并获取总结
        base64_image = encode_image(image_filepath)
        if base64_image:
            summary = get_image_summary(base64_image, json_data)
            if summary:
                txt_filename = os.path.splitext(filename)[0] + ".txt"
                txt_filepath = os.path.join(output_path, txt_filename)
                try:
                    with open(txt_filepath, "w", encoding="utf-8") as txt_file:
                        txt_file.write(summary)
                    print(f"Summary for {filename} saved to {txt_filename}")
                except Exception as e:
                    print(f"Error saving summary for {filename}: {e}")
        else:
            print(f"Skipping invalid or unsupported image file: {filename}")
    else:
        print(f"Skipping non-image file: {filename}")